- packing = False

In [1]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install unsloth

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-_2dj_i2x
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-_2dj_i2x
  Resolved https://github.com/huggingface/transformers.git to commit 74a2a4d0c790788a3b65b975510577f8dc6e81d5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.0 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.7.0.dev0-py3-none-any.whl size=11494970 sha256=bca8546cd20a32d5781a7a6f3757ebc0c1b24cdebc66aef2a0b0906c8f9b79c6
  Stored in directory: /tmp/pip-ephem-wheel-cache-2flnbpbl/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: hf-xet
    Found existin

In [ ]:
import wandb
# login
wandb_api_key = ""
wandb.login(key=wandb_api_key)

# import os
# # 현재 프로세스가 메인(첫 번째) GPU인지 확인
# is_main_process = os.environ.get("RANK", "0") == "0"
# if is_main_process:
#     import wandb
#     wandb.init(project="my_medical_soap_project", name="ddp_run_v1")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: leehh3208 (kiki44444) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# The path for the dataset for training 
shuffled_dataset_path = "/kaggle/input/datasets/hyeonhuilee2/medsynth-preprocessed/medsynth_preprocessed_shuffled"
base_model_name = "/kaggle/input/models/barnobarno/qwen3.5-4b/transformers/unsloth/1"
checkpoint_path = "./qwen3.5-4B_SFT_med"
new_model_path = "./qwen3.5-4B_SFT"
max_seq_length = 2048

# Model Loading
- Method 1: unsloth lib: FastLanguageModel
- Method 2: unsloth lib: FastVisionModel

    - I chose method 2. Because it is the newest method they released. With this, you do not need to mention which layer should be updated. It will select the right layers as requested.
    - Especially, in our case using Qwen, which is multimodal, i think this method would suit.

In [4]:
import os
# before import torch
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"                    # only if you are using more than 1 GPU
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # Prevents memory fragmentation and OOM
import torch
import unsloth
from unsloth import FastLanguageModel, FastVisionModel
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from datasets import load_from_disk


# ==========================================================
# 1. FastLanguageModel for model loading
# ==========================================================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True, # for QLoRA
    full_finetuning = False,
    random_state = 42,
)

# 2. LoRA config
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                 # ref: unsloth FT guide line
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,        # ref: unsloth FT guide line
    lora_dropout = 0,       # Unsloth is optimized the best when it's 0
    bias = "none",          # ref: unsloth FT guide line
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = max_seq_length
)

# ==========================================================
# 2. FastVisionModel for model loading
# ==========================================================

# model, tokenizer = FastVisionModel.from_pretrained(
#     base_model_name,
#     load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
#     use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
# )

# model = FastVisionModel.get_peft_model(
#     model,
#     finetune_vision_layers     = False, # False if not finetuning vision layers
#     finetune_language_layers   = True, # False if not finetuning language layers
#     finetune_attention_modules = True, # False if not finetuning attention layers
#     finetune_mlp_modules       = True, # False if not finetuning MLP layers

#     r = 16,           # The larger, the higher the accuracy, but might overfit
#     lora_alpha = 16,  # Recommended alpha == r at least
#     lora_dropout = 0,
#     bias = "none",
#     random_state = 42,
#     use_rslora = False,  # We support rank stabilized LoRA
#     loftq_config = None, # And LoftQ
#     # target_modules = "all-linear", # Optional now! Can specify a list if needed
# )

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.7: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

# Dataset formatting & preparation

This is a process where you make your dataset suitable for model input. Since we have 2 columns, Dialogue and Note, we need to combine two columns + instruction into 1 input,but in the right format for Qwen3.5. That is, in a way that Qwen3.5 can recognize instruction part, label part, and training part.

In [5]:
def format_chatml(example):
    """
    Formatting raw dataset into chat template so that your model can process it.
    
    Args:
        example (dict): Sample dataset in a dict form
    Return:
        dict: Returns the formatted result as a dictionary with the key 'text'
    """
    # Instruction
    # instruction = (
    #   "Task: Convert the medical dialogue into a professional SOAP note.\n"
    #   "Guidelines:\n"
    #   "1. STRUCTURE:\n"
    #   " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
    #   " - Objective: Observable and measurable findings from physical examination.\n"
    #   " - Assessment: Clinical diagnosis and medical reasoning.\n"
    #   " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
    #   "2. CONSTRAINTS:\n"
    #   " - Use formal clinical terminology.\n"
    #   " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
    #   " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
    # )
    instruction = (
    "Convert the medical dialogue into a SOAP note "
    "(Subjective, Objective, Assessment, Plan). "
    "Use only the provided information.\n"
    )
    dialogue = example.get("Dialogue", "")
    note = example.get(" Note", "")
    
    messages = [
        {"role": "system", "content": instruction},             # Setting persona (instruction)
        {"role": "user", "content": f"Dialogue:\n{dialogue}"},  # User's query (dialogue)
        {"role": "assistant", "content": note}                  # Ground truth (soap format)
    ]

    """apply_chat_templat :
    # Every model has its own language for separating speakers
    # (ex) using special tokens like <|im_start|> and <|im_end|>
    # This function automatically injects these tokens so you dont have to do manually
    
    add_generation_prompt=False:
    Since this is training not inference, you need to add the final trigger that
    tells the model to start generating.
    tokenize=False:
    Thie tells the tokenizer to return a string(text) rathter than a list of integers.
    """
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

In [6]:
# Dataset preparation
shuffled_ds = load_from_disk(shuffled_dataset_path)
shuffled_ds.save_to_disk("/kaggle/working/medsynth_preprocessed_shuffled")
shuffled_ds = load_from_disk("/kaggle/working/medsynth_preprocessed_shuffled")
total_count = len(shuffled_ds)
formatted_ds = shuffled_ds.map(format_chatml, remove_columns=shuffled_ds.column_names)
split_ds = formatted_ds.train_test_split(test_size=0.15, seed=42)
train_ds = split_ds['train']
eval_ds = split_ds['test']


print(f"# Train data: {len(train_ds)}")
print(f"# Eval data: {len(eval_ds)}")
print("Trainig and evaluation dataset ready")

Saving the dataset (0/1 shards):   0%|          | 0/10033 [00:00<?, ? examples/s]

Map:   0%|          | 0/10033 [00:00<?, ? examples/s]

# Train data: 8528
# Eval data: 1505
Trainig and evaluation dataset ready


# Training

## Sanity check

In [7]:
# # For sanity check setting
# # train_ds = train_ds.select(range(40))
# eval_ds = eval_ds.select(range(10))

# # ========== FastLanguageModel for model loading ==========
# from trl import SFTTrainer, SFTConfig


# # os.environ["WANDB_PROJECT"] = "Dial2Note"
# # os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # 체크포인트를 WandB에 저장 (선택 사항)


# FastLanguageModel.for_training(model) # Enable for training!
# # 5. Training

# args = SFTConfig(
#     output_dir=checkpoint_path,
    
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,
#     gradient_checkpointing=True,
#     per_device_eval_batch_size = 2,         # 1보다 훨씬 빠르고, 8보다 안전함!
#     eval_accumulation_steps = 1,            # 데이터가 1,500개여도 GPU 메모리는 평온함

#     warmup_steps=1,                         # For sanity check
#     # warmup_steps=50,
#     num_train_epochs = 2,
#     max_steps = 4,                          # for sanity check
#     # max_steps=-1,
#     max_seq_length = max_seq_length,

#     max_grad_norm = 1.0,                    # prevent loss spike: gradient clipping
#     learning_rate = 1e-5,                   # unsloth FT guideline :default
#     logging_steps = 1,                      # For sanity check
#     # logging_steps = 50,
#     optim = "adamw_8bit",                   # unsloth FT guideline
#     weight_decay = 0.001,                   # unsloth FT guideline : penalty for regularization
#     lr_scheduler_type = "linear",           # unsloth FT guideline
#     seed = 42,
#     report_to = "none",
#     # report_to = "wandb",
#     # run_name = "qwen3.5-4b_sanitycheck",        # 실험 회차 이름
#     dataset_num_proc = 2,                   # unsloth FT guideline
    
#     eval_strategy = "steps",
#     eval_steps = 3,
#     # save_strategy = "steps",
#     # save_steps = 200,
#     # save_total_limit=1,
#     # load_best_model_at_end=True,
#     # metric_for_best_model="eval_loss",
#     # greater_is_better = False,
    
# )


# # 5. Trainer config (SFTTrainer)
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     train_dataset = train_ds,
#     eval_dataset = eval_ds,
#     dataset_text_field = "text",
#     args = args,
#     # callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
# )

# print("Testing has been started. GOOD LUCK!")
# trainer.train()

In [8]:
# # @title Show final memory and time stats
# used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
# used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
# used_percentage = round(used_memory / max_memory * 100, 3)
# lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
# print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
# print(
#     f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
# )
# print(f"Peak reserved memory = {used_memory} GB.")
# print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
# print(f"Peak reserved memory % of max memory = {used_percentage} %.")
# print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Real

In [9]:
# ========== FastLanguageModel for model loading ==========
from trl import SFTTrainer, SFTConfig


os.environ["WANDB_PROJECT"] = "Dial2Note"
# os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # 체크포인트를 WandB에 저장 (선택 사항)


FastLanguageModel.for_training(model) # Enable for training!
# 5. Training

args = SFTConfig(
    # ddp_find_unused_parameters = False,
    output_dir=checkpoint_path,
    
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    per_device_eval_batch_size = 2,
    eval_accumulation_steps = 1,

    warmup_steps=50,
    num_train_epochs = 2,
    max_steps=-1,
    max_seq_length = max_seq_length,

    max_grad_norm = 1.0,                    # prevent loss spike: gradient clipping
    learning_rate = 1e-5,                   # unsloth FT guideline :default
    logging_steps = 5,
    optim = "adamw_8bit",                   # unsloth FT guideline
    weight_decay = 0.001,                   # unsloth FT guideline : penalty for regularization
    lr_scheduler_type = "linear",           # unsloth FT guideline
    seed = 42,
    report_to = "wandb",
    run_name = "qwen3.5-4b_without_DDP",        # 실험 회차 이름
    dataset_num_proc = 2,                   # unsloth FT guideline
    
    eval_strategy = "steps",
    eval_steps = 10,
    save_strategy = "steps",
    save_steps = 200,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better = False,
    
)


# 5. Trainer config (SFTTrainer)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
)


# # 5. 실행 및 저장
# if is_main_process:
#     print("Training has started. GOOD LUCK!")

trainer.train(resume_from_checkpoint = "/kaggle/input/datasets/ilovesamir/qwen3-5-4b/qwen3.5-4B_SFT_med/checkpoint-1000")

# # 학습 종료 후 동기화 및 저장
# if dist.is_initialized():
#     dist.barrier() # 모든 GPU가 끝날 때까지 대기

# # 2. 메인 프로세스(Rank 0)에서만 저장 실행
# if is_main_process:
#     print("--- Saving the final best model ---")
    
#     # 방법 A: Trainer의 기능을 이용해 모델과 토크나이저를 한꺼번에 저장 (권장)
#     trainer.save_model(new_model_path)
    
#     print(f"Model successfully saved to: {new_model_path}")

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8528 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1505 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
	logging_steps: 5 (from args) != 50 (from trainer_state.json)
	eval_steps: 10 (from args) != 200 (from trainer_state.json)
	per_device_train_batch_size: 4 (from args) != 8 (from trainer_state.json)
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,528 | Num Epochs = 2 | Total steps = 1,066
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 21,233,664 of 4,560,499,200 (0.47% trained)
wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings wit

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
1066,0.711143,0.713932


Unsloth: Restored added_tokens_decoder metadata in ./qwen3.5-4B_SFT_med/checkpoint-1066/tokenizer_config.json.


TrainOutput(global_step=1066, training_loss=0.044159712979314925, metrics={'train_runtime': 14894.8528, 'train_samples_per_second': 1.145, 'train_steps_per_second': 0.072, 'total_flos': 8.08364288401453e+17, 'train_loss': 0.044159712979314925, 'epoch': 2.0})

In [10]:
# 6. Model saving
trainer.save_model(new_model_path)
tokenizer.save_pretrained(new_model_path)
print(f"Model path: {new_model_path} saved!")

Unsloth: Restored added_tokens_decoder metadata in ./qwen3.5-4B_SFT/tokenizer_config.json.


Model path: ./qwen3.5-4B_SFT saved!


In [11]:
# ========== FastVisionModel for model loading ==========
# FastVisionModel.for_training(model) # Enable for training!
# args = SFTConfig(
#         per_device_train_batch_size = 4,
#         gradient_accumulation_steps = 8,
#         gradient_checkpointing=True,
#         per_device_eval_batch_size = 1,      # 1보다 훨씬 빠르고, 8보다 안전함!
#         eval_accumulation_steps = 1,         # 데이터가 1,500개여도 GPU 메모리는 평온함

#         warmup_steps = 5,
#         num_train_epochs = 2,
#         max_steps = -1,
#         learning_rate = 2e-4,
#         logging_steps = 80,
#         optim = "adamw_8bit",
#         weight_decay = 0.001,
#         lr_scheduler_type = "linear",
#         seed = 42,
#         output_dir = checkpoint_path,
#         report_to = "none",     # For Weights and Biases

#         # saving!
#         eval_strategy="steps",
#         eval_steps=80,
#         save_strategy="steps",
#         save_steps=80,
#         load_best_model_at_end=True,
#         metric_for_best_model="eval_loss",
#         save_total_limit=1,
        # average_tokens_across_devices=False

        # You MUST put the below items for vision finetuning:
#         remove_unused_columns = False,
#         dataset_text_field = "",
#         dataset_kwargs = {"skip_prepare_dataset": True},
#         max_length = 2048,
# )
# trainer = SFTTrainer(
#     model = model,
#     tokenizer = tokenizer,
#     # data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
#     dataset_text_field = "text",
#     train_dataset = train_ds,
#     eval_dataset = eval_ds,
#     args = args,
#     callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]
# )

In [12]:
# instruction = (
#     "Task: Convert the medical dialogue into a professional SOAP note.\n"
#     "Guidelines:\n"
#     "1. STRUCTURE:\n"
#     " - Subjective: Patient-reported information (CC, HPI, ROS).\n"
#     " - Objective: Observable and measurable findings from physical examination.\n"
#     " - Assessment: Clinical diagnosis and medical reasoning.\n"
#     " - Plan: Treatment recommendations, prescriptions, and follow-up care.\n"
#     "2. CONSTRAINTS:\n"
#     " - Use formal clinical terminology.\n"
#     " - Use ONLY facts from the dialogue. DO NOT invent names, ages, or data.\n"
#     " - Match gender pronouns and anatomical lateralization (Right/Left) strictly.\n"
# )

# example_dialogue = """
# [Doctor]: Hello there. What brings you in today?

# [Patient]: Hi, doctor. Two days ago, I tripped while going down the stairs, and since then, my right ankle has been really painful and swollen.

# [Doctor]: I’m sorry to hear that. Does the pain get worse when you try to walk?

# [Patient]: Yes, it’s not too bad when I’m resting, but as soon as I put weight on it or try to walk, the pain is much sharper.

# [Doctor]: I see. Let me take a look at the ankle. (After a brief examination) I see some bruising and swelling around the outer ankle bone here. Does it hurt when I press this spot?

# [Patient]: Ow! Yes, that really hurts. Do you think it’s broken?

# [Doctor]: It’s hard to tell for sure right now. Your vitals—blood pressure and temperature—look normal, but because you have tenderness and swelling over the lateral malleolus, we need to rule out a fracture.

# [Patient]: Okay. I haven't had any fever or anything, but the pain is the main issue. What’s the plan?

# [Doctor]: First, I’ll prescribe Ibuprofen 400mg for the pain. You can take it every 6 hours as needed. Also, I’m ordering a 3-view X-ray of your right ankle to get a better look at the bone.

# [Patient]: That makes sense. Should I come back here after the X-ray?

# [Doctor]: Exactly. Once the X-ray results are ready, we’ll review them together and decide on the next steps for your treatment.
# """

# example_note = (
#     "1. **Subjective:**\n\n"
#     "**Chief Complaint (CC):**\n"
#     "- Right ankle pain and swelling.\n\n"
#     "**History of Present Illness (HPI):**\n"
#     "- The patient presents with right ankle pain and swelling following a trip on the stairs 2 days ago. The pain is described as sharp upon weight-bearing or walking and improves with rest. There is no history of fever or similar prior injuries.\n\n"
#     "**Review of Systems (ROS):**\n"
#     "- Musculoskeletal: Positive for right ankle pain, swelling, and bruising.\n"
#     "- General: Negative for fever or chills.\n"
#     "- Cardiovascular: No complaints.\n"
#     "- Respiratory: No complaints.\n\n"
#     "2. **Objective:**\n\n"
#     "**Vital Signs:**\n"
#     "- BP: 120/80 mmHg\n"
#     "- HR: 72 bpm\n"
#     "- RR: 18 breaths/min\n"
#     "- Temp: 98.6°F\n"
#     "- SpO2: 98% on room air\n\n"
#     "**Physical Examination:**\n"
#     "- Musculoskeletal: Bruising and swelling noted around the outer ankle. Tenderness localized over the right lateral malleolus.\n\n"
#     "3. **Assessment:**\n\n"
#     "**Diagnosis:**\n"
#     "- Acute right ankle sprain. Rule out lateral malleolus fracture due to significant tenderness and swelling.\n\n"
#     "4. **Plan:**\n\n"
#     "**Medical Management:**\n"
#     "- Ibuprofen 400 mg orally every 6 hours as needed for pain.\n\n"
#     "**Investigations Ordered:**\n"
#     "- X-ray of the right ankle (3 views) to rule out fracture.\n\n"
#     "**Follow-Up:**\n"
#     "- The patient is instructed to return to the clinic for a review of X-ray results and to determine further treatment steps."
# )




# def get_predictions(model, tokenizer, dialogue_list, batch_size=8):
#     FastVisionModel.for_inference(model) # Enable for inference!
    
#     all_predictions = []
#     # few_shot_user = f"Dialogue:\n{example_dialogue}"
#     # few_shot_assistant = example_note
#     anchored_text = "1. **Subjective:**\n\n**Chief Complaint (CC):**\n-"
#     for i in range(0, len(dialogue_list), batch_size):
#         batch_texts = dialogue_list[i : i + batch_size]

#         prompts = []
#         for d in batch_texts:
#             messages = [
#                 {"role": "system", "content": instruction}, # 이전에 수정한 상세 지시문
    
#                 # Few-shot
#                 # {"role": "user", "content": few_shot_user},
#                 # {"role": "assistant", "content": few_shot_assistant},
    
#                 {"role": "user", "content": f"Dialogue:\n{d}"}
#             ]
#             # add_generation_prompt=True: automatically produces <|im_start|>assistant\n
#             prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#             prompt += anchored_text
#             prompts.append(prompt)
        
#         inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        
#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=1024,
#                 do_sample=False,
#                 pad_token_id=tokenizer.pad_token_id,
#                 eos_token_id=tokenizer.eos_token_id # EOS 명시
#             )
        
#         input_len = inputs.input_ids.shape[1]
#         final_outputs = outputs[:, input_len:]
        
#         decoded_outputs = tokenizer.batch_decode(final_outputs, skip_special_tokens=True)
#         # all_predictions.extend([p.strip() for p in decoded_outputs])
#         for p in decoded_outputs:
#           full_note = anchored_text + p
#           all_predictions.append(full_note.strip())
        
        
#     return all_predictions

In [13]:
# test_ds = shuffled_ds.select(range(total_count-10, total_count))
# dials = test_ds['Dialogue'][:1]
# references = test_ds[' Note'][:1]
# predictions = get_predictions(model, tokenizer, dials, batch_size=32)
# print("Finished generation")